# Notebook 03 — Baseline Recommender

**Purpose:** Implement and evaluate a strong baseline model that will serve as a benchmark and a fallback system for all subsequent models.

**Inputs:**
- `data/splits/train.csv` — historical ratings (train only)
- `data/splits/test.csv` — future ratings (evaluation only)

**Outputs:**
- Rating metrics: RMSE, MAE
- Ranking metrics: Precision@10, NDCG@10
- Popularity ranking table
- Saved model artifact: `results/baseline_model.pkl`

## 0. Imports & Configuration

Set all constants here. No magic numbers in the code below.

In [1]:
import sys
import os
import numpy as np
import pandas as pd
import joblib

# Add src/ to path so we can import project modules
sys.path.insert(0, os.path.abspath("../src"))

from baseline import BaselineModel
from evaluation import (
    evaluate_rating_predictions,
    evaluate_ranking,
    run_sanity_checks,
)

# ── Paths ──────────────────────────────────────────────────────
TRAIN_PATH   = "../data/splits/train.csv"
TEST_PATH    = "../data/splits/test.csv"
RESULTS_DIR  = "../results/"

# ── Hyperparameters ───────────────────────────────────────────
POPULARITY_THRESHOLD = 50   # Minimum vote count (m) for Bayesian weighted score
TOP_K                = 10   # Number of recommendations per user
RELEVANCE_THRESHOLD  = 3.5  # Minimum rating to count a test item as "relevant"

# ── Sanity check bounds ───────────────────────────────────────
RMSE_RANGE = (0.8, 1.5)     # Expected RMSE range for a valid baseline
MAE_RANGE  = (0.6, 1.2)     # Expected MAE range for a valid baseline

os.makedirs(RESULTS_DIR, exist_ok=True)
print("Configuration loaded.")

Configuration loaded.


## 1. Load Data

Load train and test sets from disk. This is the ONLY time data is read. Train is used for fitting; test is used only for evaluation at the end.

In [2]:
train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)

print(f"Train: {len(train):,} rows | {train['userId'].nunique():,} users | {train['movieId'].nunique():,} movies")
print(f"Test : {len(test):,} rows  | {test['userId'].nunique():,} users | {test['movieId'].nunique():,} movies")
print()
print("Train rating stats:")
print(train["rating"].describe().round(4))

Train: 1,565,182 rows | 12,773 users | 14,246 movies
Test : 397,622 rows  | 12,773 users | 13,902 movies

Train rating stats:
count    1.565182e+06
mean     3.569400e+00
std      1.058200e+00
min      5.000000e-01
25%      3.000000e+00
50%      4.000000e+00
75%      4.000000e+00
max      5.000000e+00
Name: rating, dtype: float64


## 2. Fit the Baseline Model

Call `fit()` with `train` only. Internally this computes:
- μ (global mean)
- b_u per user
- b_i per item
- Popularity ranking via Bayesian weighted score

In [3]:
model = BaselineModel(popularity_threshold=POPULARITY_THRESHOLD)
model.fit(train)

print(f"\nGlobal mean (μ) = {model.mu:.4f}")

[BaselineModel.fit] Done.
  mu              = 3.5694
  unique users    = 12,773
  unique items    = 14,246
  popularity rows = 14,246

Global mean (μ) = 3.5694


## 3. Generate Predictions on Test Set

Vectorised batch prediction. All (userId, movieId) pairs in test receive a prediction. Unknown users or items automatically fall back to the bias hierarchy.

In [4]:
y_pred = model.predict_batch(test)

# Quick spot-check — should have same row count as test, no NaNs
assert len(y_pred) == len(test), f"Prediction count mismatch: {len(y_pred)} vs {len(test)}"
assert not np.isnan(y_pred).any(), "NaN predictions detected — check fallback logic"

print(f"Generated {len(y_pred):,} predictions")
print(f"Prediction range: [{y_pred.min():.2f}, {y_pred.max():.2f}]")
print(f"Prediction mean : {y_pred.mean():.4f}  (expected ≈ {model.mu:.4f})")

Generated 397,622 predictions
Prediction range: [0.50, 5.00]
Prediction mean : 3.5005  (expected ≈ 3.5694)


## 4. Rating Prediction Evaluation (RMSE & MAE)

Evaluate point-wise accuracy. Both metrics compare predicted ratings vs the actual ratings in test.csv.

In [5]:
rating_metrics = evaluate_rating_predictions(test, y_pred)

print("=" * 40)
print("  RATING PREDICTION METRICS")
print("=" * 40)
print(f"  RMSE : {rating_metrics['rmse']:.4f}")
print(f"  MAE  : {rating_metrics['mae']:.4f}")
print(f"  Rows : {rating_metrics['n_preds']:,}")
print(f"  NaN  : {rating_metrics['n_nan_preds']}")

  RATING PREDICTION METRICS
  RMSE : 0.9047
  MAE  : 0.6826
  Rows : 397,622
  NaN  : 0


## 5. Sanity Checks

Validates: no NaN predictions, all values in [0.5, 5.0], RMSE and MAE within expected ranges. ALL checks must pass before we proceed.

In [6]:
checks = run_sanity_checks(
    y_pred,
    rmse=rating_metrics["rmse"],
    mae=rating_metrics["mae"],
    rmse_range=RMSE_RANGE,
    mae_range=MAE_RANGE,
)

print("=" * 40)
print("  SANITY CHECKS")
print("=" * 40)
all_passed = True
for c in checks:
    status = "✅ PASS" if c["passed"] else "❌ FAIL"
    print(f"  {status}  {c['check']}  —  {c['detail']}")
    if not c["passed"]:
        all_passed = False

print()
if all_passed:
    print("  ✅ All checks passed. Baseline is valid.")
else:
    print("  ❌ Some checks FAILED. Investigate before proceeding.")
    raise AssertionError("Baseline sanity checks failed — model is not reliable.")

  SANITY CHECKS
  ✅ PASS  No NaN predictions  —  0 NaN values found
  ✅ PASS  All predictions in [0.5, 5.0]  —  0 out-of-range values
  ✅ PASS  RMSE in (0.8, 1.5)  —  RMSE = 0.9047
  ✅ PASS  MAE in (0.6, 1.2)  —  MAE = 0.6826

  ✅ All checks passed. Baseline is valid.


## 6. Fallback Usage Diagnostics

Report how many test rows triggered the fallback (unknown users or items). This is expected to be low since the temporal split guarantees all test users exist in train.

In [7]:
stats = model.fallback_stats(test)

print("=" * 40)
print("  FALLBACK / COLD-START DIAGNOSTICS")
print("=" * 40)
print(f"  Total test rows    : {stats['total_test_rows']:,}")
print(f"  Unseen users       : {stats['unseen_users']:,}  ({stats['pct_unseen_users']}%)")
print(f"  Unseen items       : {stats['unseen_items']:,}  ({stats['pct_unseen_items']}%)")
print(f"  Both unseen        : {stats['both_unseen']:,}  ({stats['pct_both_unseen']}%)")
print()
print("Leakage check: μ was computed from train ONLY → NO leakage.")

  FALLBACK / COLD-START DIAGNOSTICS
  Total test rows    : 397,622
  Unseen users       : 0  (0.0%)
  Unseen items       : 321  (0.08%)
  Both unseen        : 0  (0.0%)

Leakage check: μ was computed from train ONLY → NO leakage.


## 7. Popularity Ranking (Top-K Recommendations)

Evaluate ranking quality via Precision@10 and NDCG@10. Items the user already rated in train are masked out before ranking. 'Relevant' is defined as rating >= 3.5.

In [8]:
# Define a wrapper that matches the interface expected by evaluate_ranking
def popularity_recommend(userId, k, seen_items):
    top_k_df = model.recommend_top_k(userId, k=k, seen_items=seen_items)
    return top_k_df["movieId"].tolist()

ranking_metrics = evaluate_ranking(
    test=test,
    train=train,
    recommend_fn=popularity_recommend,
    k=TOP_K,
    relevance_threshold=RELEVANCE_THRESHOLD,
)

print("=" * 40)
print("  RANKING METRICS (Popularity Baseline)")
print("=" * 40)
print(f"  Precision@{TOP_K}  : {ranking_metrics['precision_at_k']:.4f}")
print(f"  NDCG@{TOP_K}       : {ranking_metrics['ndcg_at_k']:.4f}")
print(f"  Users evaluated : {ranking_metrics['n_users_evaluated']:,}")
print(f"  Relevance ≥ {RELEVANCE_THRESHOLD}")

  RANKING METRICS (Popularity Baseline)
  Precision@10  : 0.0389
  NDCG@10       : 0.0498
  Users evaluated : 12,623
  Relevance ≥ 3.5


## 8. Top-20 Popular Movies

Inspect the popularity ranking. The weighted score shrinks movies with few votes toward the global mean (Bayesian regularisation).

In [9]:
top20 = model.popularity_df.head(20).copy()
top20.index = range(1, len(top20) + 1)
top20.index.name = "rank"

print(f"Popularity threshold (m) = {POPULARITY_THRESHOLD}")
print(f"Global mean (μ)          = {model.mu:.4f}")
print()
top20[["movieId", "R", "v", "score"]].rename(
    columns={"R": "mean_rating", "v": "n_ratings", "score": "weighted_score"}
).round(4)

Popularity threshold (m) = 50
Global mean (μ)          = 3.5694



,movieId,mean_rating,n_ratings,weighted_score
rank,,,,
1,318,4.4167,6000,4.4097
2,858,4.3225,3712,4.3125
3,50,4.2996,3752,4.2900
4,202439,4.3586,488,4.2853
5,1221,4.2746,2365,4.2600
6,1203,4.2803,1115,4.2498
7,2959,4.2523,4387,4.2446
8,750,4.2592,1732,4.2399
9,5618,4.2508,1812,4.2325


## 9. Final Summary

Collect all results in a single dictionary and save the fitted model to disk for use by subsequent notebooks.

In [10]:
# ── Section 9: Sample User Output — Baseline Recommendations ──────────────────
# Shows train history + Popularity Top-K for 3 representative users.
# Uses: `model`, `train`, `test` (already in memory above)

import ast

# Load movie metadata for readable titles and genres
_meta_raw = pd.read_csv("../data/raw/movies_metadata.csv",
                         usecols=["id", "title", "genres"], low_memory=False)
_meta_raw["id"] = pd.to_numeric(_meta_raw["id"], errors="coerce")
_meta_raw = _meta_raw.dropna(subset=["id"])
_meta_raw["id"] = _meta_raw["id"].astype(int)

# Parse the genres JSON column → "Action|Comedy" string
def _parse_genres(g):
    try:
        return "|".join(d["name"] for d in ast.literal_eval(g))
    except Exception:
        return "Unknown"
_meta_raw["genre_str"] = _meta_raw["genres"].apply(_parse_genres)
_meta = _meta_raw.rename(columns={"id": "tmdbId"})[["tmdbId", "title", "genre_str"]]

# Join via links.csv (movieId → tmdbId)
_links = pd.read_csv("../data/raw/links.csv", usecols=["movieId", "tmdbId"])
_links["tmdbId"] = pd.to_numeric(_links["tmdbId"], errors="coerce")
_links = _links.dropna(subset=["tmdbId"])
_links["tmdbId"] = _links["tmdbId"].astype(int)
_movie_info = _links.merge(_meta, on="tmdbId", how="left").set_index("movieId")

def _movie_label(movie_id):
    if movie_id in _movie_info.index:
        row = _movie_info.loc[movie_id]
        return row["title"], row["genre_str"]
    return f"movieId={movie_id}", "Unknown"

# Pick 3 users: sparse, medium, active
_train_counts = train.groupby("userId").size()
_all_users    = _train_counts.index.tolist()
_sparse  = _train_counts.idxmin()
_medium  = _train_counts[((_train_counts > 20) & (_train_counts < 60))].index[0]
_active  = _train_counts.idxmax()
SAMPLE_USERS = [_sparse, _medium, _active]

SEP = "─" * 65

def show_baseline_user(uid):
    user_train = train[train["userId"] == uid].sort_values("timestamp")
    seen       = set(user_train["movieId"])

    label_map = {4: "cold-start", 10: "sparse", 50: "moderate", 100: "active"}
    n = len(user_train)
    activity = next((v for k, v in sorted(label_map.items()) if n <= k), "power user")

    print(f"\nUser {uid}:")
    print(SEP)
    print(f"Interactions in train: {n}  ({activity})")
    print("Movies rated:")
    for _, row in user_train.iterrows():
        title, genres = _movie_label(row["movieId"])
        print(f"  - {title} | Rating: {row['rating']} | Genres: {genres}")

    print(f"\nBaseline (Popularity) Recommendations  [Top-{TOP_K}]:")
    recs_df = model.recommend_top_k(uid, k=TOP_K, seen_items=seen)
    for rank, (_, r) in enumerate(recs_df.iterrows(), 1):
        title, genres = _movie_label(r["movieId"])
        print(f"  {rank}. {title}")
        print(f"     Genres: {genres} | Weighted score: {r['score']:.4f}")
    print()

for uid in SAMPLE_USERS:
    show_baseline_user(uid)


User 5392:
─────────────────────────────────────────────────────────────────
Interactions in train: 12  (moderate)
Movies rated:
  - My Fair Lady | Rating: 4.5 | Genres: Drama|Family|Music|Romance
  - Batman & Robin | Rating: 3.5 | Genres: Action|Crime|Fantasy
  - Lethal Weapon 3 | Rating: 4.0 | Genres: Adventure|Action|Comedy|Thriller|Crime
  - Time Bandits | Rating: 3.5 | Genres: Family|Fantasy|Science Fiction|Adventure|Comedy
  - Austin Powers in Goldmember | Rating: 5.0 | Genres: Comedy|Crime|Science Fiction
  - I Know What You Did Last Summer | Rating: 3.5 | Genres: Horror|Thriller|Mystery
  - American Graffiti | Rating: 2.0 | Genres: Comedy|Drama
  - JFK | Rating: 3.5 | Genres: Drama|Thriller|History
  - The Meaning of Life | Rating: 2.0 | Genres: Comedy
  - Tank Girl | Rating: 3.0 | Genres: Action|Comedy|Fantasy|Science Fiction
  - Evil Dead II | Rating: 0.5 | Genres: Horror|Comedy|Fantasy
  - An American Werewolf in London | Rating: 3.0 | Genres: Horror|Comedy

Baseline (Popul

  - ¡Three Amigos! | Rating: 5.0 | Genres: Comedy|Western
  - Bad Boys | Rating: 4.0 | Genres: Action|Comedy|Crime|Thriller
  - A Life Less Ordinary | Rating: 3.0 | Genres: Fantasy|Drama|Comedy|Thriller|Science Fiction|Romance
  - Lethal Weapon 4 | Rating: 4.0 | Genres: Action|Adventure|Comedy|Crime|Thriller
  - Outrageous Fortune | Rating: 2.0 | Genres: Action|Comedy|Mystery
  - Fright Night | Rating: 3.0 | Genres: Horror
  - Broken Arrow | Rating: 3.0 | Genres: Action|Adventure|Drama|Thriller
  - The Greatest Show on Earth | Rating: 3.0 | Genres: Action|Drama|Romance
  - Gremlins | Rating: 3.0 | Genres: Fantasy|Horror|Comedy
  - Weird Science | Rating: 3.0 | Genres: Comedy|Romance|Fantasy
  - Blast from the Past | Rating: 4.0 | Genres: Comedy|Romance
  - Piranha | Rating: 2.0 | Genres: Horror
  - Revenge of the Nerds | Rating: 3.0 | Genres: Comedy
  - The Englishman Who Went Up a Hill But Came Down a Mountain | Rating: 5.0 | Genres: Drama|Comedy|Romance
  - Phenomenon | Rating: 5.0 |

  - The Unforgiven | Rating: 4.0 | Genres: Romance|Western|Drama
  - Mo' Better Blues | Rating: 3.0 | Genres: Drama|Romance
  - Plunkett & MacLeane | Rating: 2.5 | Genres: Drama|Action|Comedy
  - Just Visiting | Rating: 2.0 | Genres: Comedy|Fantasy|Science Fiction
  - The Visitors | Rating: 2.0 | Genres: Fantasy|Comedy|Science Fiction
  - Drumline | Rating: 3.5 | Genres: Drama|Romance|Comedy|Music
  - Maid in Manhattan | Rating: 2.0 | Genres: Comedy|Drama|Romance
  - The Crying Game | Rating: 4.0 | Genres: Romance|Crime|Drama|Thriller
  - This Is My Father | Rating: 2.5 | Genres: Drama|Romance
  - George Washington | Rating: 3.0 | Genres: Drama
  - The Recruit | Rating: 3.5 | Genres: Action|Thriller
  - Down to Earth | Rating: 3.5 | Genres: Fantasy|Comedy|Science Fiction|Romance
  - Two Weeks Notice | Rating: 3.5 | Genres: Comedy|Romance
  - A Midsummer Night's Dream | Rating: 3.5 | Genres: Fantasy|Drama|Comedy|Romance
  - The Flight of the Phoenix | Rating: 4.0 | Genres: Adventure|Dra

  - Random Hearts | Rating: 2.5 | Genres: Drama|Romance
  - Edmond | Rating: 2.5 | Genres: Drama|Thriller
  - Pumpkin | Rating: 2.5 | Genres: Comedy|Drama|Romance
  - The Sisterhood of the Traveling Pants | Rating: 2.5 | Genres: Drama|Comedy
  - Venus | Rating: 4.5 | Genres: Drama|Comedy|Romance
  - Los Olvidados | Rating: 3.5 | Genres: Crime|Drama
  - Casino Royale | Rating: 3.5 | Genres: Adventure|Action|Thriller
  - Harsh Times | Rating: 3.0 | Genres: Crime|Drama|Thriller|Action
  - Normal | Rating: 4.0 | Genres: Drama
  - The Bad Sleep Well | Rating: 3.0 | Genres: Crime|Drama|Thriller
  - Becket | Rating: 4.5 | Genres: Drama|History
  - Next Friday | Rating: 3.0 | Genres: Comedy
  - Tokyo Story | Rating: 3.5 | Genres: Drama
  - Bridge to Terabithia | Rating: 2.5 | Genres: Adventure|Drama|Family
  - The Painted Veil | Rating: 3.5 | Genres: Drama|Romance
  - P.S. | Rating: 3.0 | Genres: Comedy|Drama|Fantasy|Romance
  - Once | Rating: 4.5 | Genres: Drama|Music|Romance
  - Save the Tig

In [11]:
summary = {
    "model":           "BaselineModel (Bias + Popularity)",
    "global_mean_mu":  round(model.mu, 4),
    "n_train_users":   len(model.user_bias),
    "n_train_items":   len(model.item_bias),
    "n_test_rows":     len(test),
    "rmse":            rating_metrics["rmse"],
    "mae":             rating_metrics["mae"],
    f"precision_at_{TOP_K}": ranking_metrics["precision_at_k"],
    f"ndcg_at_{TOP_K}":      ranking_metrics["ndcg_at_k"],
    "unseen_users":     stats["unseen_users"],
    "unseen_items":     stats["unseen_items"],
    "no_leakage":       True,
    "no_nan_preds":     rating_metrics["n_nan_preds"] == 0,
}

print("=" * 50)
print("  FINAL SUMMARY — BASELINE MODEL")
print("=" * 50)
for k, v in summary.items():
    print(f"  {k:<30s} : {v}")

# Save fitted model for downstream notebooks
model_path = os.path.join(RESULTS_DIR, "baseline_model.pkl")
joblib.dump(model, model_path)
print(f"\n✅ Model saved → {model_path}")

  FINAL SUMMARY — BASELINE MODEL
  model                          : BaselineModel (Bias + Popularity)
  global_mean_mu                 : 3.5694
  n_train_users                  : 12773
  n_train_items                  : 14246
  n_test_rows                    : 397622
  rmse                           : 0.90471
  mae                            : 0.682559
  precision_at_10                : 0.038881
  ndcg_at_10                     : 0.049819
  unseen_users                   : 0
  unseen_items                   : 321
  no_leakage                     : True
  no_nan_preds                   : True

✅ Model saved → ../results/baseline_model.pkl
